# 🌲 Calibration Tree - Precision Resistances

## 🎯 Objective

Cross-calibration of **4 precision resistance sets** through offset chaining.

## 📊 Structure

- **Temperature references (not used as raised)**: Sensors 1009 and 1010 (channels 13 and 14)
  - Used internally within each set for offset calculation
  - NOT used to chain offsets between different sets
- **First round (R1)**: 4 resistance sets
  - RESIST_SET1: PDHD-HP-13 to PDHD-HP-24
  - RESIST_SET2: PDHD-HP-25 to PDHD-HP-36
  - RESIST_SET3: PDHD-HP-37 to PDHD-HP-48
  - RESIST_SET4: PDHD-HP-49 to PDHD-HP-60
- **Second round (R2)**: 1 set with 3 resistances from each R1 set
  - RESIST_SET5: Mix of 12 resistances (3 from each R1 set)

## ⚙️ Method

1. **Per-set processing**: Calculate offsets and constants within each set
2. **Identify "raised sensors"**: Resistance sensors that appear in both R1 and R2
3. **Offset chaining**: Connect R1 resistances through R2 using raised sensors as bridges
4. **Final calibration constants**: Calculate offsets between any two resistances
   - Same set: Direct offset calculation (using internal reference)
   - Different sets: Multi-path calculation via R2 raised sensors
   - Weighted average: Weight = 1/error² (gives more importance to precise paths)

## 🔧 Classes used

- `SetSTS`: Adapted version for resistances
- `RunSTS`: Individual run handling
- `CalibrationNetwork`: Calibration graph construction (optional)

In [1]:
# =============================================================================
# SETUP E IMPORTS
# =============================================================================

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Add paths
project_path = os.path.abspath("../../")
sys.path.append(project_path)
src_dir = os.path.abspath("../src")
sys.path.append(src_dir)

print("🔍 Paths configurados:")
print(f"  - Project path: {project_path}")
print(f"  - Src dir: {src_dir}")

# Imports
try:
    import importlib
    if 'RTD_Calibration_VGP.src.calibration_network' in sys.modules:
        importlib.reload(sys.modules['RTD_Calibration_VGP.src.calibration_network'])
    
    from RTD_Calibration_VGP.src.calibration_network import CalibrationNetwork
    from RTD_Calibration_VGP.src.setSTS import SetSTS
    from RTD_Calibration_VGP.src.logfile import Logfile
    print("✅ Imports desde RTD_Calibration_VGP.src completados")
except ImportError as e:
    print(f"⚠️ Error importando desde RTD_Calibration_VGP.src: {e}")
    try:
        from calibration_network import CalibrationNetwork
        from setSTS import SetSTS
        from logfile import Logfile
        print("✅ Imports locales completados")
    except ImportError as e2:
        print(f"❌ Error importando clases: {e2}")
        raise e2

print("✅ Setup completado")
print("📁 Directorio de trabajo:", os.getcwd())

🔍 Paths configurados:
  - Project path: /Users/vicky/Desktop/rtd-calibration-ana
  - Src dir: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/src
✅ Imports desde RTD_Calibration_VGP.src completados
✅ Setup completado
📁 Directorio de trabajo: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/notebooks


## 📂 Carga de Datos

In [2]:
# ------------------------------------------------------------------
# CARGAR LOGFILE
# ------------------------------------------------------------------
print("="*80)
print("📂 CARGANDO LOGFILE")
print("="*80)

logfile_paths = [
    "../data/LogFile.csv",
    "RTD_Calibration_VGP/data/LogFile.csv",
    "../../data/LogFile.csv"
]

logfile = None
for path in logfile_paths:
    if os.path.exists(path):
        try:
            logfile = Logfile(path)
            print(f"✅ Logfile cargado desde {path}")
            print(f"   Registros totales: {len(logfile.log_file)}")
            break
        except Exception as e:
            print(f"⚠️ Error: {e}")

if logfile is None:
    raise FileNotFoundError("No se pudo encontrar el logfile")

# Filtrar solo RESIST_SET
resist_data = logfile.log_file[
    logfile.log_file['CalibSetNumber'].astype(str).str.contains('RESIST_SET', na=False)
]
print(f"\n📊 Registros de RESIST_SET: {len(resist_data)}")
print(f"   Sets encontrados: {sorted(resist_data['CalibSetNumber'].unique())}")

📂 CARGANDO LOGFILE
CSV file loaded successfully from '../data/LogFile.csv'.
✅ Logfile cargado desde ../data/LogFile.csv
   Registros totales: 832

📊 Registros de RESIST_SET: 21
   Sets encontrados: ['RESIST_SET0', 'RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4', 'RESIST_SET5']


## ⚙️ Round Configuration

We manually define the rounds since we have a simple tree structure:
- **Round 1**: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4
- **Round 2**: RESIST_SET5 (contains 3 resistances from each R1 set)

In [3]:
# ------------------------------------------------------------------
# ROUND CONFIGURATION
# ------------------------------------------------------------------

# Manual mapping of sets to rounds
sets_config = {
    'RESIST_SET1': {'round': 1, 'description': 'PDHD-HP-13 to PDHD-HP-24'},
    'RESIST_SET2': {'round': 1, 'description': 'PDHD-HP-25 to PDHD-HP-36'},
    'RESIST_SET3': {'round': 1, 'description': 'PDHD-HP-37 to PDHD-HP-48'},
    'RESIST_SET4': {'round': 1, 'description': 'PDHD-HP-49 to PDHD-HP-60'},
    'RESIST_SET5': {'round': 2, 'description': 'Mix: 3 from each R1 set'}
}

# NOTE: SetSTS uses channel 2 as internal reference to calculate offsets within each set
# The sensor in channel 2 varies depending on the set.
# 
# Temperature sensors 1009 and 1010 are present in channels 13 and 14,
# and are NOT used to chain offsets between different sets.
# 
# Chaining R1 → R2 is done through RAISED SENSORS
# (resistance sensors that appear in both R1 and R2)

print("⚙️ CONFIGURATION:")
print("   📌 Internal reference: Channel 2 (varies per set)")
print("   🔗 Offset chaining: Via RAISED resistance sensors (R1 ∩ R2)")
print("   ⚠️  Temperature sensors 1009/1010 (ch 13-14) are NOT used for chaining")
print("\n   Sets per round:")
for ronda in [1, 2]:
    sets_in_round = [s for s, cfg in sets_config.items() if cfg['round'] == ronda]
    print(f"   - Round {ronda}: {sets_in_round}")

⚙️ CONFIGURATION:
   📌 Internal reference: Channel 2 (varies per set)
   🔗 Offset chaining: Via RAISED resistance sensors (R1 ∩ R2)
   ⚠️  Temperature sensors 1009/1010 (ch 13-14) are NOT used for chaining

   Sets per round:
   - Round 1: ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']
   - Round 2: ['RESIST_SET5']


## 🔄 Procesamiento de Sets

In [4]:
# ------------------------------------------------------------------
# PROCESAR TODOS LOS RESIST_SET
# ------------------------------------------------------------------
print("="*80)
print("🔄 PROCESAMIENTO DE SETS")
print("="*80)

processed_sets = {}
all_sensors = set()

# Crear una única instancia de SetSTS para todos los RESIST_SET
print("\n📦 Inicializando SetSTS para resistencias...")
try:
    set_sts = SetSTS(
        logfile=logfile.log_file,
        data_folder="resistences"
    )
    print("✅ SetSTS inicializado")
except Exception as e:
    print(f"❌ Error inicializando SetSTS: {e}")
    import traceback
    traceback.print_exc()
    raise

# Agrupar todos los runs por set
print("\n🔄 Agrupando runs por set...")
try:
    set_sts.group_runs_by_set(calibset_pattern="RESIST_SET")
    print(f"✅ Runs agrupados: {len(set_sts.runs_by_set)} sets encontrados")
    print(f"   Sets: {list(set_sts.runs_by_set.keys())}")
except Exception as e:
    print(f"❌ Error agrupando runs: {e}")
    import traceback
    traceback.print_exc()
    raise

# Calcular offsets y RMS para todos los sets a la vez
print("\n📊 Calculando offsets y RMS...")
try:
    # Solo procesar los sets que están en nuestra configuración
    selected_sets = list(sets_config.keys())
    set_sts.calculate_offsets_and_rms(selected_sets=selected_sets, tini=20, tend=40)
    print("✅ Offsets y RMS calculados")
except Exception as e:
    print(f"❌ Error calculando offsets: {e}")
    import traceback
    traceback.print_exc()

# Calcular repetibilidad y estadísticas globales
print("\n📈 Calculando repetibilidad y estadísticas globales...")
try:
    set_sts.offset_repeatability(
        tini=20, 
        tend=40, 
        selected_sets=selected_sets,
        save_dir="offset_repeatability_resistences",
        write_csv=False,  # No guardar CSVs intermedios
        write_excel=False  # No guardar Excel intermedios
    )
    print(f"✅ Estadísticas calculadas para {len(set_sts.global_stats)} sets")
except Exception as e:
    print(f"❌ Error calculando repetibilidad: {e}")
    import traceback
    traceback.print_exc()

# Extraer información de cada set procesado
for set_name in sets_config.keys():
    print(f"\n{'='*60}")
    print(f"📦 Extrayendo datos de {set_name}")
    print(f"{'='*60}")
    
    if set_name not in set_sts.runs_by_set:
        print(f"⚠️ {set_name} no encontrado en runs agrupados")
        continue
    
    if set_name not in set_sts.global_stats:
        print(f"⚠️ {set_name} no tiene estadísticas calculadas")
        continue
    
    try:
        runs_dict = set_sts.runs_by_set[set_name]
        stats = set_sts.global_stats[set_name]
        
        print(f"   ✅ Runs: {len(runs_dict)}")
        print(f"   🔍 Claves en stats: {list(stats.keys())}")
        
        # IMPORTANTE: setSTS guarda las estadísticas con claves 'means' y 'sigmas' (en mK)
        if 'means' in stats and stats['means']:
            means_dict = stats['means']  # Diccionario {sensor_id: mean_offset_mK}
            sigmas_dict = stats.get('sigmas', {})  # Diccionario {sensor_id: sigma_mK}
            
            print(f"   📊 Medias encontradas: {len(means_dict)} sensores")
            print(f"      Ejemplo: {list(means_dict.items())[:3]}")
            
            # Convertir diccionarios a Series de pandas (sensor_id como índice)
            # Y convertir de mK a K
            offset_means = pd.Series(means_dict) / 1000.0  # mK → K
            offset_stds = pd.Series(sigmas_dict) / 1000.0 if sigmas_dict else pd.Series()
            
            # IMPORTANTE: Los sensor IDs son strings alfanuméricos (ej: 'PDHD-HP-25')
            # NO convertir a int, mantenerlos como strings
            sensors_in_set = offset_means.index.tolist()
            
            # FILTRAR sensores de temperatura (1009, 1010) que están en canales 13-14
            # Estos NO son resistencias, solo referencias internas
            sensors_in_set = [s for s in sensors_in_set if s not in ['1009', '1010']]
            
            all_sensors.update(sensors_in_set)
            
            processed_sets[set_name] = {
                'set_obj': set_sts,  # Guardamos referencia al objeto completo
                'round': sets_config[set_name]['round'],
                'sensors': sensors_in_set,
                'n_runs': len(runs_dict),
                'offset_means': offset_means,
                'offset_stds': offset_stds,
                'runs_dict': runs_dict  # Guardar también los runs para CalibrationNetwork
            }
            
            print(f"   📊 Sensores procesados: {len(sensors_in_set)}")
            print(f"      IDs: {sensors_in_set[:5]}{'...' if len(sensors_in_set) > 5 else ''}")
            print(f"   📏 Offsets (K): min={offset_means.min():.6f}, max={offset_means.max():.6f}")
        else:
            print(f"   ⚠️ No hay 'means' en estadísticas o está vacío")
            
    except Exception as e:
        print(f"   ❌ Error extrayendo datos de {set_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"📊 RESUMEN DE PROCESAMIENTO")
print(f"{'='*80}")
print(f"   Sets procesados exitosamente: {len(processed_sets)}")
print(f"   Total de sensores únicos: {len(all_sensors)}")
if len(all_sensors) <= 20:
    print(f"   Sensores: {sorted(all_sensors)}")
else:
    sensors_sorted = sorted(all_sensors)
    print(f"   Sensores: {sensors_sorted[:10]} ... {sensors_sorted[-5:]}")

🔄 PROCESAMIENTO DE SETS

📦 Inicializando SetSTS para resistencias...
✅ SetSTS inicializado

🔄 Agrupando runs por set...

🔄 Processing CalibSetNumber: RESIST_SET0
  ❌ Excluded: 20251016_air_HP13_HP14_PDHD-HP-1-PDHD-HP-12_1_pre (contains excluded keyword)

🔄 Processing CalibSetNumber: RESIST_SET1
Archivo de temperatura encontrado: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET1/20251023_air_STSr10_STSr9_PDHD-HP-13-PDHD-HP-24_1.txt
Valores NaN (contador): 0
Empty DataFrame
Columns: [datetime, channel, value]
Index: []
Archivo de temperatura procesado correctamente: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET1/20251023_air_STSr10_STSr9_PDHD-HP-13-PDHD-HP-24_1.txt
No se detectaron canales defectuosos.
Valores de sensores extraídos (antes de filtrado y conversión): ['PDHD-HP-13' 'PDHD-HP-14' 'PDHD-HP-15' 'PDHD-HP-16' 'PDHD-HP-17'
 'PDHD-HP-18' 'PDHD-HP-19' 'PDHD-HP-20' '

## 🔧 Añadir Sensores de Referencia (Canal 2)

El sensor del canal 2 de cada set se usa como referencia interna para calcular offsets, por lo que **no aparece** en las estadísticas (su offset respecto a sí mismo sería 0).

Sin embargo, **necesitamos incluirlo** para poder calcular offsets entre él y otros sensores. Lo añadimos manualmente con offset = 0 y error = 0.

In [15]:
# ------------------------------------------------------------------
# AÑADIR SENSORES DEL CANAL 2 (REFERENCIA INTERNA)
# ------------------------------------------------------------------
print("="*80)
print("🔧 AÑADIENDO SENSORES DE REFERENCIA (CANAL 2)")
print("="*80)

# El sensor del canal 2 es la referencia interna y no aparece en offset_means
# Lo añadimos manualmente con offset=0 y error=0 para poder calcular offsets con él

# Mapeo de sets a sus sensores del canal 2 (basado en LogFile)
channel_2_sensors = {
    'RESIST_SET1': 'PDHD-HP-14',  # Canal 2 de SET1
    'RESIST_SET2': 'PDHD-HP-26',  # Canal 2 de SET2
    'RESIST_SET3': 'PDHD-HP-38',  # Canal 2 de SET3
    'RESIST_SET4': 'PDHD-HP-50',  # Canal 2 de SET4
    'RESIST_SET5': 'PDHD-HP-19'   # Verificar cuál es el del SET5
}

print("\n📋 Sensores del canal 2 por set:")
for set_name, sensor_ch2 in channel_2_sensors.items():
    print(f"   {set_name}: {sensor_ch2}")

print("\n🔄 Añadiendo sensores del canal 2 a processed_sets...")

for set_name in processed_sets.keys():
    if set_name not in channel_2_sensors:
        print(f"⚠️ No se conoce el sensor del canal 2 para {set_name}")
        continue
    
    sensor_ch2 = channel_2_sensors[set_name]
    
    # Verificar si ya está (no debería)
    if sensor_ch2 in processed_sets[set_name]['offset_means'].index:
        print(f"✅ {set_name}: {sensor_ch2} ya está presente")
        continue
    
    # Añadir con offset=0 y error=0 (es la referencia)
    processed_sets[set_name]['offset_means'][sensor_ch2] = 0.0
    processed_sets[set_name]['offset_stds'][sensor_ch2] = 0.0
    processed_sets[set_name]['sensors'].append(sensor_ch2)
    all_sensors.add(sensor_ch2)
    
    # Ordenar la lista de sensores para mantener el orden correcto
    processed_sets[set_name]['sensors'].sort()
    
    print(f"✅ {set_name}: Añadido {sensor_ch2} (offset=0, error=0)")

print(f"\n{'='*80}")
print("✅ SENSORES DE REFERENCIA AÑADIDOS")
print(f"{'='*80}")
print(f"   Total sensores únicos ahora: {len(all_sensors)}")

# Mostrar resumen actualizado
print("\n📊 Sensores por set (actualizado):")
for set_name in sorted(processed_sets.keys()):
    n_sensors = len(processed_sets[set_name]['sensors'])
    print(f"   {set_name}: {n_sensors} sensores")


🔧 AÑADIENDO SENSORES DE REFERENCIA (CANAL 2)

📋 Sensores del canal 2 por set:
   RESIST_SET1: PDHD-HP-14
   RESIST_SET2: PDHD-HP-26
   RESIST_SET3: PDHD-HP-38
   RESIST_SET4: PDHD-HP-50
   RESIST_SET5: PDHD-HP-19

🔄 Añadiendo sensores del canal 2 a processed_sets...
✅ RESIST_SET1: PDHD-HP-14 ya está presente
✅ RESIST_SET2: PDHD-HP-26 ya está presente
✅ RESIST_SET3: PDHD-HP-38 ya está presente
✅ RESIST_SET4: PDHD-HP-50 ya está presente
✅ RESIST_SET5: PDHD-HP-19 ya está presente

✅ SENSORES DE REFERENCIA AÑADIDOS
   Total sensores únicos ahora: 48

📊 Sensores por set (actualizado):
   RESIST_SET1: 12 sensores
   RESIST_SET2: 12 sensores
   RESIST_SET3: 12 sensores
   RESIST_SET4: 12 sensores
   RESIST_SET5: 12 sensores


## 🔗 Identificación de Sensores "Raised"

Los sensores "raised" son aquellos que aparecen en múltiples rondas y permiten encadenar offsets.

In [16]:
# ------------------------------------------------------------------
# IDENTIFICAR SENSORES RAISED
# ------------------------------------------------------------------
print("="*80)
print("🔗 IDENTIFICACIÓN DE SENSORES RAISED")
print("="*80)

# Agrupar sensores por ronda
sensors_by_round = {}
for set_name, data in processed_sets.items():
    ronda = data['round']
    if ronda not in sensors_by_round:
        sensors_by_round[ronda] = set()
    sensors_by_round[ronda].update(data['sensors'])

print("\n📊 Sensores por ronda:")
for ronda in sorted(sensors_by_round.keys()):
    sensors = sorted(sensors_by_round[ronda])
    print(f"   Ronda {ronda}: {len(sensors)} sensores")
    print(f"      {sensors}")

# Identificar raised (sensores que aparecen en R1 y R2)
# ESTOS son los sensores que usaremos para encadenar offsets entre rondas
if 1 in sensors_by_round and 2 in sensors_by_round:
    raised_sensors = sensors_by_round[1].intersection(sensors_by_round[2])
    
    print(f"\n🔗 Sensores RAISED (R1→R2): {len(raised_sensors)}")
    print(f"   {sorted(raised_sensors)}")
    print(f"\n   ✅ Estos sensores se usarán para encadenar offsets R1 → R2")
else:
    raised_sensors = set()
    print("\n⚠️ No se encontraron sensores raised")

# Info: los sensores 1009 y 1010 están presentes como referencia interna (canal 2)
# pero NO participan en el encadenamiento de offsets
print(f"\n📌 Nota: Sensores 1009 y 1010 (STSr9 y STSr10)")
print("   Función: Referencias internas en cada set (canal 2)")
print("   NO se usan para encadenar offsets entre rondas")

🔗 IDENTIFICACIÓN DE SENSORES RAISED

📊 Sensores por ronda:
   Ronda 1: 48 sensores
      ['PDHD-HP-13', 'PDHD-HP-14', 'PDHD-HP-15', 'PDHD-HP-16', 'PDHD-HP-17', 'PDHD-HP-18', 'PDHD-HP-19', 'PDHD-HP-20', 'PDHD-HP-21', 'PDHD-HP-22', 'PDHD-HP-23', 'PDHD-HP-24', 'PDHD-HP-25', 'PDHD-HP-26', 'PDHD-HP-27', 'PDHD-HP-28', 'PDHD-HP-29', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-32', 'PDHD-HP-33', 'PDHD-HP-34', 'PDHD-HP-35', 'PDHD-HP-36', 'PDHD-HP-37', 'PDHD-HP-38', 'PDHD-HP-39', 'PDHD-HP-40', 'PDHD-HP-41', 'PDHD-HP-42', 'PDHD-HP-43', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-46', 'PDHD-HP-47', 'PDHD-HP-48', 'PDHD-HP-49', 'PDHD-HP-50', 'PDHD-HP-51', 'PDHD-HP-52', 'PDHD-HP-53', 'PDHD-HP-54', 'PDHD-HP-55', 'PDHD-HP-56', 'PDHD-HP-57', 'PDHD-HP-58', 'PDHD-HP-59', 'PDHD-HP-60']
   Ronda 2: 12 sensores
      ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

🔗 Sensores RAISED (R1→R2): 12
   ['PDH

## ✅ Verificación de Datos Procesados

In [17]:
# ------------------------------------------------------------------
# VERIFICAR DATOS PROCESADOS
# ------------------------------------------------------------------
print("="*80)
print("✅ VERIFICACIÓN DE DATOS PROCESADOS")
print("="*80)

print("\n📊 Resumen de sets procesados:")
for set_name in sorted(processed_sets.keys()):
    data = processed_sets[set_name]
    print(f"\n{set_name} (Ronda {data['round']}):")
    print(f"   Sensores: {len(data['sensors'])}")
    print(f"   Runs: {data['n_runs']}")
    print(f"   Offsets calculados: Sí")
    
    # Mostrar los primeros sensores, indicando cuál es el canal 2 (referencia)
    # NOTA: 1009 y 1010 ya fueron filtrados en la celda de procesamiento
    if set_name in channel_2_sensors:
        ch2_sensor = channel_2_sensors[set_name]
        primeros = data['sensors'][:13]
        sensores_str = []
        for s in primeros:
            if s == ch2_sensor:
                sensores_str.append(f"{s} (canal 2 - ref)")
            else:
                sensores_str.append(s)
        print(f"   Primeros sensores: {', '.join(sensores_str[:4])}")
    else:
        print(f"   Primeros sensores: {', '.join(data['sensors'][:4])}")

print("\n✅ Datos listos para calcular constantes de calibración")
print("   (Los sensores raised se identificarán en la siguiente celda)")

✅ VERIFICACIÓN DE DATOS PROCESADOS

📊 Resumen de sets procesados:

RESIST_SET1 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculados: Sí
   Primeros sensores: PDHD-HP-13, PDHD-HP-14 (canal 2 - ref), PDHD-HP-15, PDHD-HP-16

RESIST_SET2 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculados: Sí
   Primeros sensores: PDHD-HP-25, PDHD-HP-26 (canal 2 - ref), PDHD-HP-27, PDHD-HP-28

RESIST_SET3 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculados: Sí
   Primeros sensores: PDHD-HP-37, PDHD-HP-38 (canal 2 - ref), PDHD-HP-39, PDHD-HP-40

RESIST_SET4 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculados: Sí
   Primeros sensores: PDHD-HP-49, PDHD-HP-50 (canal 2 - ref), PDHD-HP-51, PDHD-HP-52

RESIST_SET5 (Ronda 2):
   Sensores: 12
   Runs: 4
   Offsets calculados: Sí
   Primeros sensores: PDHD-HP-13, PDHD-HP-19 (canal 2 - ref), PDHD-HP-21, PDHD-HP-28

✅ Datos listos para calcular constantes de calibración
   (Los sensores raised se identificarán en la siguiente celda)


## ⛓️ Offsets entre Resistencias usando Sensores Raised

Calculamos offsets entre resistencias de diferentes sets de R1 usando los **3 sensores raised** de cada set como puentes.

**Método**: Para cada par de resistencias, calculamos 3 offsets (uno por cada sensor raised) y promediamos pesando por el error.

In [18]:
# ------------------------------------------------------------------
# IDENTIFICAR SENSORES RAISED POR SET DE R1
# ------------------------------------------------------------------
print("="*80)
print("🔗 IDENTIFICACIÓN DE SENSORES RAISED POR SET R1")
print("="*80)

# Los sensores raised son aquellos que aparecen tanto en R1 como en R2
# (ya fueron filtrados 1009 y 1010 en el procesamiento inicial)

print(f"\n📊 Total sensores raised: {len(raised_sensors)}")
print(f"   IDs: {sorted(raised_sensors)}")

print(f"\n📋 RESIST_SET5 (R2) contiene:")
if 'RESIST_SET5' in processed_sets:
    set5_sensors = processed_sets['RESIST_SET5']['sensors']
    print(f"   {len(set5_sensors)} sensores: {set5_sensors}")
else:
    print("   ⚠️ RESIST_SET5 no procesado")

# Identificar cuáles sensores raised pertenecen a cada set de R1
raised_by_set = {}  # {set_name: [lista de sensores raised]}

for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
    if set_name not in processed_sets:
        continue
    
    # Sensores en este set de R1
    sensors_r1 = set(processed_sets[set_name]['sensors'])
    
    # Intersección con sensores raised
    raised_in_this_set = sorted(sensors_r1.intersection(raised_sensors))
    raised_by_set[set_name] = raised_in_this_set
    
    print(f"\n📦 {set_name}:")
    print(f"   Total sensores: {len(sensors_r1)}")
    print(f"   Sensores raised: {len(raised_in_this_set)}")
    print(f"   IDs: {raised_in_this_set}")

print(f"\n{'='*80}")
print("✅ Identificación completa")
print(f"   Total sensores raised: {len(raised_sensors)}")
print(f"   Distribución: SET1:{len(raised_by_set['RESIST_SET1'])}, SET2:{len(raised_by_set['RESIST_SET2'])}, SET3:{len(raised_by_set['RESIST_SET3'])}, SET4:{len(raised_by_set['RESIST_SET4'])}")
print(f"   Estos sensores conectan R1 con R2 (RESIST_SET5)")

🔗 IDENTIFICACIÓN DE SENSORES RAISED POR SET R1

📊 Total sensores raised: 12
   IDs: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📋 RESIST_SET5 (R2) contiene:
   12 sensores: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📦 RESIST_SET1:
   Total sensores: 12
   Sensores raised: 3
   IDs: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']

📦 RESIST_SET2:
   Total sensores: 12
   Sensores raised: 3
   IDs: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

📦 RESIST_SET3:
   Total sensores: 12
   Sensores raised: 3
   IDs: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

📦 RESIST_SET4:
   Total sensores: 12
   Sensores raised: 3
   IDs: ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

✅ Identificación completa
   Total sensores raised: 12
   Distribución: SET1:3, SET2:3,

## 🎯 Cálculo de Offsets entre Sets usando Sensores Raised

Calculamos offsets entre resistencias de **diferentes sets** de R1 usando sensores raised como puentes.

**Estrategia**: 
1. Calcular offset de resistencia A a cada sensor raised de su set
2. Calcular offset de resistencia B a cada sensor raised de su set
3. Usar los sensores raised comunes para obtener 3 caminos independientes
4. Promediar los 3 offsets pesando por el error (media ponderada)

In [31]:
# ------------------------------------------------------------------
# FUNCIÓN PARA CALCULAR OFFSET ENTRE SETS USANDO SENSORES RAISED
# ------------------------------------------------------------------

def calculate_offset_via_raised(set_A, sensor_A, set_B, sensor_B, verbose=False):
    """
    Calcula el offset entre dos resistencias de diferentes sets usando sensores raised.
    
    Usa los 3 sensores raised de cada set como puentes:
    - Calcula 3 offsets independientes (uno por cada sensor raised)
    - Promedia usando media ponderada (peso = 1/error²)
    
    Args:
        set_A: Nombre del set de la resistencia A (ej: 'RESIST_SET1')
        sensor_A: ID de la resistencia A (ej: 'PDHD-HP-15')
        set_B: Nombre del set de la resistencia B (ej: 'RESIST_SET2')
        sensor_B: ID de la resistencia B (ej: 'PDHD-HP-27')
        verbose: Si True, imprime información detallada
        
    Returns:
        tuple: (offset_promedio, error_promedio) o (None, None) si hay error
    """
    if set_A not in processed_sets or set_B not in processed_sets:
        if verbose:
            print(f"⚠️ Sets no encontrados")
        return None, None
    
    if set_A not in raised_by_set or set_B not in raised_by_set:
        if verbose:
            print(f"⚠️ No hay sensores raised identificados")
        return None, None
    
    # Obtener offsets y errores de cada set
    offset_means_A = processed_sets[set_A]['offset_means']
    offset_stds_A = processed_sets[set_A]['offset_stds']
    offset_means_B = processed_sets[set_B]['offset_means']
    offset_stds_B = processed_sets[set_B]['offset_stds']
    
    # Verificar que los sensores existen
    if sensor_A not in offset_means_A.index or sensor_B not in offset_means_B.index:
        if verbose:
            print(f"⚠️ Sensores no encontrados en sus sets")
        return None, None
    
    # Sensores raised de cada set
    raised_list_A = raised_by_set[set_A]
    raised_list_B = raised_by_set[set_B]
    
    if verbose:
        print(f"\n🔗 Calculando offset {sensor_A} ({set_A}) → {sensor_B} ({set_B})")
        print(f"   Sensores raised en {set_A}: {raised_list_A}")
        print(f"   Sensores raised en {set_B}: {raised_list_B}")
    
    # Necesitamos offsets de R2 para conectar los sensores raised
    if 'RESIST_SET5' not in processed_sets:
        if verbose:
            print(f"   ⚠️ RESIST_SET5 (R2) no está disponible")
        return None, None
    
    offset_means_R2 = processed_sets['RESIST_SET5']['offset_means']
    offset_stds_R2 = processed_sets['RESIST_SET5']['offset_stds']
    
    # Calcular offsets para cada par de sensores raised (uno de cada set)
    offsets_per_path = []
    errors_per_path = []
    paths_info = []
    
    for raised_A in raised_list_A:
        if raised_A not in offset_means_R2.index:
            continue  # El sensor raised de A debe estar en R2
        
        for raised_B in raised_list_B:
            if raised_B not in offset_means_R2.index:
                continue  # El sensor raised de B debe estar en R2
            
            # CAMINO: sensor_A → raised_A (en set_A) → raised_A (en R2) → raised_B (en R2) → raised_B (en set_B) → sensor_B
            
            # Paso 1: Offset de sensor_A a raised_A en set A
            offset_A_to_raisedA = offset_means_A[sensor_A] - offset_means_A[raised_A]
            error_A_to_raisedA = np.sqrt(
                (offset_stds_A[sensor_A] if sensor_A in offset_stds_A.index else 0.0)**2 +
                (offset_stds_A[raised_A] if raised_A in offset_stds_A.index else 0.0)**2
            )
            
            # Paso 2: Offset de raised_A a raised_B en R2
            offset_raisedA_to_raisedB_R2 = offset_means_R2[raised_A] - offset_means_R2[raised_B]
            error_raisedA_to_raisedB_R2 = np.sqrt(
                (offset_stds_R2[raised_A] if raised_A in offset_stds_R2.index else 0.0)**2 +
                (offset_stds_R2[raised_B] if raised_B in offset_stds_R2.index else 0.0)**2
            )
            
            # Paso 3: Offset de raised_B a sensor_B en set B
            offset_raisedB_to_B = offset_means_B[raised_B] - offset_means_B[sensor_B]
            error_raisedB_to_B = np.sqrt(
                (offset_stds_B[raised_B] if raised_B in offset_stds_B.index else 0.0)**2 +
                (offset_stds_B[sensor_B] if sensor_B in offset_stds_B.index else 0.0)**2
            )
            
            # Offset total por este camino: suma algebraica de los 3 pasos
            offset_via_this_path = offset_A_to_raisedA + offset_raisedA_to_raisedB_R2 + offset_raisedB_to_B
            # Error total: propagación en cuadratura (suma cuadrática)
            error_via_this_path = np.sqrt(error_A_to_raisedA**2 + error_raisedA_to_raisedB_R2**2 + error_raisedB_to_B**2)
            
            offsets_per_path.append(offset_via_this_path)
            errors_per_path.append(error_via_this_path)
            paths_info.append((raised_A, raised_B))
            
            if verbose:
                print(f"\n   📍 Camino via {raised_A} ↔ {raised_B}:")
                print(f"      Paso 1: {sensor_A} → {raised_A} (en {set_A}): {offset_A_to_raisedA:+.6f} ± {error_A_to_raisedA:.6f} K")
                print(f"      Paso 2: {raised_A} → {raised_B} (en R2): {offset_raisedA_to_raisedB_R2:+.6f} ± {error_raisedA_to_raisedB_R2:.6f} K")
                print(f"      Paso 3: {raised_B} → {sensor_B} (en {set_B}): {offset_raisedB_to_B:+.6f} ± {error_raisedB_to_B:.6f} K")
                print(f"      → Total camino: ({offset_A_to_raisedA:+.6f}) + ({offset_raisedA_to_raisedB_R2:+.6f}) + ({offset_raisedB_to_B:+.6f}) = {offset_via_this_path:+.6f} K")
                print(f"      → Error camino: √({error_A_to_raisedA:.6f}² + {error_raisedA_to_raisedB_R2:.6f}² + {error_raisedB_to_B:.6f}²) = {error_via_this_path:.6f} K")
    
    if len(offsets_per_path) == 0:
        if verbose:
            print(f"   ⚠️ No hay sensores raised comunes entre los dos sets")
        return None, None
    
    # Calcular media ponderada (peso = 1/error²)
    weights = np.array([1.0 / (err**2) if err > 0 else 1e6 for err in errors_per_path])
    offsets_array = np.array(offsets_per_path)
    
    offset_weighted = np.sum(offsets_array * weights) / np.sum(weights)
    error_weighted = np.sqrt(1.0 / np.sum(weights))
    
    if verbose:
        print(f"\n   {'─'*70}")
        print(f"   📊 MEDIA PONDERADA (de {len(offsets_per_path)} caminos independientes):")
        print(f"   {'─'*70}")
        print(f"\n   Caminos disponibles: {len(raised_list_A)} (set A) × {len(raised_list_B)} (set B) = {len(offsets_per_path)} caminos")
        
        # Identificar el camino con menor error (más preciso)
        idx_best = np.argmin(errors_per_path)
        best_error = errors_per_path[idx_best]
        best_offset = offsets_array[idx_best]
        best_path = paths_info[idx_best]
        
        print(f"\n   🏆 Camino más preciso: {best_path[0]} ↔ {best_path[1]}")
        print(f"      Error: {best_error:.6f} K (el más bajo)")
        print(f"      Offset: {best_offset:+.6f} K")
        
        print(f"\n   Fórmula de media ponderada:")
        print(f"      Offset_final = Σ(offset_i × peso_i) / Σ(peso_i)")
        print(f"      donde peso_i = 1 / error_i²")
        print(f"\n   Cálculo de pesos:")
        for i, (offset_i, error_i, (rA, rB)) in enumerate(zip(offsets_array, errors_per_path, paths_info)):
            peso_i = weights[i]
            marker = " 🏆" if i == idx_best else ""
            print(f"      Camino {i+1} ({rA}↔{rB}): peso = 1/{error_i:.6f}² = {peso_i:.2f}{marker}")
        
        suma_pesos = np.sum(weights)
        print(f"\n   Suma total de pesos: Σ(peso_i) = {suma_pesos:.2f}")
        
        print(f"\n   Cálculo del numerador (Σ offset_i × peso_i):")
        numerador = 0.0
        for i, (offset_i, peso_i, (rA, rB)) in enumerate(zip(offsets_array, weights, paths_info)):
            contrib = offset_i * peso_i
            numerador += contrib
            print(f"      Camino {i+1}: {offset_i:+.6f} × {peso_i:.2f} = {contrib:+.4f}")
        print(f"      → Numerador total = {numerador:+.4f}")
        
        print(f"\n   Offset final = {numerador:+.4f} / {suma_pesos:.2f} = {offset_weighted:+.6f} K")
        print(f"\n   Error final = √(1 / Σ peso_i) = √(1 / {suma_pesos:.2f}) = {error_weighted:.6f} K")
        print(f"\n   🎯 RESULTADO:")
        print(f"      Offset: {offset_weighted:+.6f} ± {error_weighted:.6f} K")
        print(f"      (promedio de {len(offsets_per_path)} mediciones independientes, ponderado por precisión)")
    
    return offset_weighted, error_weighted

print("="*80)
print("🔧 FUNCIÓN calculate_offset_via_raised() DEFINIDA")
print("="*80)
print("   Calcula offsets entre resistencias de diferentes sets")
print("   Usa los 3 sensores raised de cada set como puentes")
print("   Promedia con media ponderada (peso = 1/error²)")

🔧 FUNCIÓN calculate_offset_via_raised() DEFINIDA
   Calcula offsets entre resistencias de diferentes sets
   Usa los 3 sensores raised de cada set como puentes
   Promedia con media ponderada (peso = 1/error²)


## 📊 Ejemplo: Offsets entre Sets Usando Sensores Raised

Demostramos el cálculo de offsets entre resistencias de diferentes sets de R1.

In [32]:
# ------------------------------------------------------------------
# EJEMPLO 1: OFFSET ENTRE RESIST_SET1 Y RESIST_SET2
# ------------------------------------------------------------------
print("="*80)
print("📊 EJEMPLO 1: Offset SET1 → SET2")
print("="*80)

# Elegir una resistencia de cada set
sensor_set1 = 'PDHD-HP-15'  # De RESIST_SET1
sensor_set2 = 'PDHD-HP-27'  # De RESIST_SET2

if ('RESIST_SET1' in processed_sets and 'RESIST_SET2' in processed_sets and
    sensor_set1 in processed_sets['RESIST_SET1']['offset_means'].index and
    sensor_set2 in processed_sets['RESIST_SET2']['offset_means'].index):
    
    offset, error = calculate_offset_via_raised(
        'RESIST_SET1', sensor_set1,
        'RESIST_SET2', sensor_set2,
        verbose=True
    )
    
    if offset is not None:
        print(f"\n{'='*80}")
        print(f"✅ RESULTADO FINAL")
        print(f"{'='*80}")
        print(f"   Offset {sensor_set1} → {sensor_set2}: {offset:+.6f} ± {error:.6f} K")
        print(f"   Equivalente: {offset*1000:+.3f} ± {error*1000:.3f} mK")
else:
    print("⚠️ Sensores no disponibles para el ejemplo")

# ------------------------------------------------------------------
# EJEMPLO 2: OFFSET ENTRE RESIST_SET3 Y RESIST_SET4
# ------------------------------------------------------------------
print(f"\n\n{'='*80}")
print("📊 EJEMPLO 2: Offset SET3 → SET4")
print("="*80)

# Elegir una resistencia de cada set
sensor_set3 = 'PDHD-HP-39'  # De RESIST_SET3
sensor_set4 = 'PDHD-HP-51'  # De RESIST_SET4

if ('RESIST_SET3' in processed_sets and 'RESIST_SET4' in processed_sets and
    sensor_set3 in processed_sets['RESIST_SET3']['offset_means'].index and
    sensor_set4 in processed_sets['RESIST_SET4']['offset_means'].index):
    
    offset, error = calculate_offset_via_raised(
        'RESIST_SET3', sensor_set3,
        'RESIST_SET4', sensor_set4,
        verbose=True
    )
    
    if offset is not None:
        print(f"\n{'='*80}")
        print(f"✅ RESULTADO FINAL")
        print(f"{'='*80}")
        print(f"   Offset {sensor_set3} → {sensor_set4}: {offset:+.6f} ± {error:.6f} K")
        print(f"   Equivalente: {offset*1000:+.3f} ± {error*1000:.3f} mK")
else:
    print("⚠️ Sensores no disponibles para el ejemplo")

# ------------------------------------------------------------------
# RESUMEN DE CONECTIVIDAD
# ------------------------------------------------------------------
print(f"\n\n{'='*80}")
print("🔗 RESUMEN DE CONECTIVIDAD ENTRE SETS")
print("="*80)

print("""
✅ Con esta función podemos calcular offsets entre CUALQUIER par de resistencias
   en diferentes sets de R1.

🔗 El método usa R2 como intermediario:
   SET_A sensor → raised_A (en SET_A) → raised_A (en R2) → raised_B (en R2) → raised_B (en SET_B) → SET_B sensor

📌 Para cada par de sets, se calculan múltiples caminos (uno por cada par de sensores raised)
   y se promedian ponderando por el error (peso = 1/error²).

🎯 Cada set de R1 tiene 2-3 sensores raised, generando hasta 9 caminos independientes.
   La media ponderada da mayor peso a los caminos más precisos.
""")

📊 EJEMPLO 1: Offset SET1 → SET2

🔗 Calculando offset PDHD-HP-15 (RESIST_SET1) → PDHD-HP-27 (RESIST_SET2)
   Sensores raised en RESIST_SET1: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Sensores raised en RESIST_SET2: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

   📍 Camino via PDHD-HP-13 ↔ PDHD-HP-28:
      Paso 1: PDHD-HP-15 → PDHD-HP-13 (en RESIST_SET1): -0.002503 ± 0.000040 K
      Paso 2: PDHD-HP-13 → PDHD-HP-28 (en R2): -0.030311 ± 0.000022 K
      Paso 3: PDHD-HP-28 → PDHD-HP-27 (en RESIST_SET2): +0.020804 ± 0.000063 K
      → Total camino: (-0.002503) + (-0.030311) + (+0.020804) = -0.012010 K
      → Error camino: √(0.000040² + 0.000022² + 0.000063²) = 0.000078 K

   📍 Camino via PDHD-HP-13 ↔ PDHD-HP-30:
      Paso 1: PDHD-HP-15 → PDHD-HP-13 (en RESIST_SET1): -0.002503 ± 0.000040 K
      Paso 2: PDHD-HP-13 → PDHD-HP-30 (en R2): +0.016052 ± 0.000027 K
      Paso 3: PDHD-HP-30 → PDHD-HP-27 (en RESIST_SET2): -0.025549 ± 0.000214 K
      → Total camino: (-0.002503) + (+0.016052) + (

In [34]:
# ------------------------------------------------------------------
# VERIFICACIÓN: ¿Cuántos caminos se calcularon?
# ------------------------------------------------------------------
print("="*80)
print("🔍 VERIFICACIÓN DE CAMINOS CALCULADOS")
print("="*80)

# Calcular un ejemplo sin verbose para contar caminos
sensor_test1 = 'PDHD-HP-15'  # SET1
sensor_test2 = 'PDHD-HP-27'  # SET2

offset_test, error_test = calculate_offset_via_raised(
    'RESIST_SET1', sensor_test1,
    'RESIST_SET2', sensor_test2,
    verbose=False
)

print(f"\n📊 Resumen del cálculo {sensor_test1} → {sensor_test2}:")
print(f"   Sensores raised en RESIST_SET1: {len(raised_by_set['RESIST_SET1'])} → {raised_by_set['RESIST_SET1']}")
print(f"   Sensores raised en RESIST_SET2: {len(raised_by_set['RESIST_SET2'])} → {raised_by_set['RESIST_SET2']}")
print(f"   Total caminos posibles: {len(raised_by_set['RESIST_SET1'])} × {len(raised_by_set['RESIST_SET2'])} = {len(raised_by_set['RESIST_SET1']) * len(raised_by_set['RESIST_SET2'])} caminos")
print(f"\n✅ Offset calculado: {offset_test:+.6f} ± {error_test:.6f} K")
print(f"   (usando media ponderada de todos los caminos disponibles)")

print(f"\n💡 Nota: El verbose=True muestra el detalle de TODOS los caminos calculados")

🔍 VERIFICACIÓN DE CAMINOS CALCULADOS

📊 Resumen del cálculo PDHD-HP-15 → PDHD-HP-27:
   Sensores raised en RESIST_SET1: 3 → ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Sensores raised en RESIST_SET2: 3 → ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']
   Total caminos posibles: 3 × 3 = 9 caminos

✅ Offset calculado: -0.012029 ± 0.000040 K
   (usando media ponderada de todos los caminos disponibles)

💡 Nota: El verbose=True muestra el detalle de TODOS los caminos calculados


In [35]:
# ------------------------------------------------------------------
# DEBUG: Verificar qué sensores raised están en R2
# ------------------------------------------------------------------
print("="*80)
print("🔍 DEBUG: Sensores raised en R2")
print("="*80)

if 'RESIST_SET5' in processed_sets:
    sensors_r2 = processed_sets['RESIST_SET5']['sensors']
    print(f"\nSensores en RESIST_SET5 (R2): {len(sensors_r2)}")
    print(f"   {sensors_r2}")
    
    print(f"\n📊 Verificación de sensores raised por set:")
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in raised_by_set:
            continue
        raised_list = raised_by_set[set_name]
        print(f"\n{set_name}: {len(raised_list)} sensores raised")
        print(f"   Lista: {raised_list}")
        
        # Verificar cuáles están en R2
        en_r2 = [s for s in raised_list if s in sensors_r2]
        no_en_r2 = [s for s in raised_list if s not in sensors_r2]
        
        print(f"   ✅ En R2 ({len(en_r2)}): {en_r2}")
        if no_en_r2:
            print(f"   ❌ NO en R2 ({len(no_en_r2)}): {no_en_r2}")


🔍 DEBUG: Sensores raised en R2

Sensores en RESIST_SET5 (R2): 12
   ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📊 Verificación de sensores raised por set:

RESIST_SET1: 3 sensores raised
   Lista: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   ✅ En R2 (3): ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']

RESIST_SET2: 3 sensores raised
   Lista: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']
   ✅ En R2 (3): ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

RESIST_SET3: 3 sensores raised
   Lista: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']
   ✅ En R2 (3): ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

RESIST_SET4: 3 sensores raised
   Lista: ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']
   ✅ En R2 (3): ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']


## 🔄 Offsets Directos entre Resistencias del Mismo Set

Calculamos offsets **directos** entre dos resistencias del mismo set de R1, **sin pasar por R2**.

Si ambas resistencias están en el mismo set, sus offsets ya están calculados respecto a la referencia interna (canal 2). Para obtener el offset entre ellas:

**Offset(A → B) = Offset(A → Ref) - Offset(B → Ref)**

In [36]:
# ------------------------------------------------------------------
# CALCULAR OFFSETS DIRECTOS ENTRE RESISTENCIAS DEL MISMO SET
# ------------------------------------------------------------------
print("="*80)
print("🔄 OFFSETS DIRECTOS ENTRE RESISTENCIAS (MISMO SET)")
print("="*80)

# Ejemplo: Calcular offset entre dos resistencias de RESIST_SET1
set_example = 'RESIST_SET1'
sensor_A = 'PDHD-HP-15'  # Primera resistencia
sensor_B = 'PDHD-HP-20'  # Segunda resistencia

print(f"\n📊 Ejemplo: Offset entre {sensor_A} y {sensor_B} en {set_example}")
print(f"{'='*80}")

if set_example in processed_sets:
    offset_means = processed_sets[set_example]['offset_means']
    offset_stds = processed_sets[set_example]['offset_stds']
    
    if sensor_A in offset_means.index and sensor_B in offset_means.index:
        # Offset de A respecto a la referencia (canal 2)
        offset_A_ref = offset_means[sensor_A]
        error_A = offset_stds[sensor_A] if sensor_A in offset_stds.index else 0.0
        
        # Offset de B respecto a la referencia (canal 2)
        offset_B_ref = offset_means[sensor_B]
        error_B = offset_stds[sensor_B] if sensor_B in offset_stds.index else 0.0
        
        # Offset DIRECTO de A a B: Offset(A→B) = Offset(A→Ref) - Offset(B→Ref)
        offset_A_to_B = offset_A_ref - offset_B_ref
        
        # Error propagado (suma cuadrática)
        error_A_to_B = np.sqrt(error_A**2 + error_B**2)
        
        print(f"\n✅ Cálculo completado:")
        print(f"   📏 {sensor_A} → Ref: {offset_A_ref:+.6f} ± {error_A:.6f} K")
        print(f"   📏 {sensor_B} → Ref: {offset_B_ref:+.6f} ± {error_B:.6f} K")
        print(f"\n🎯 Offset DIRECTO {sensor_A} → {sensor_B}:")
        print(f"   {offset_A_to_B:+.6f} ± {error_A_to_B:.6f} K")
        print(f"\n📌 Este offset NO requiere pasar por R2")
        print(f"   Se calcula directamente usando la referencia interna del set")
    else:
        print(f"\n⚠️ Uno o ambos sensores no encontrados en {set_example}")
        print(f"   Sensores disponibles: {offset_means.index.tolist()}")
else:
    print(f"\n⚠️ {set_example} no procesado")

# ------------------------------------------------------------------
# FUNCIÓN GENERAL PARA CALCULAR OFFSETS DIRECTOS
# ------------------------------------------------------------------

def calculate_direct_offset(set_name, sensor_from, sensor_to):
    """
    Calcula el offset directo entre dos sensores del mismo set.
    
    Args:
        set_name: Nombre del set (ej: 'RESIST_SET1')
        sensor_from: ID del sensor origen
        sensor_to: ID del sensor destino
        
    Returns:
        tuple: (offset, error) o (None, None) si hay error
    """
    if set_name not in processed_sets:
        print(f"⚠️ Set {set_name} no encontrado")
        return None, None
    
    offset_means = processed_sets[set_name]['offset_means']
    offset_stds = processed_sets[set_name]['offset_stds']
    
    if sensor_from not in offset_means.index or sensor_to not in offset_means.index:
        print(f"⚠️ Sensores no encontrados en {set_name}")
        return None, None
    
    # Calcular offset directo
    offset = offset_means[sensor_from] - offset_means[sensor_to]
    error = np.sqrt(
        (offset_stds[sensor_from] if sensor_from in offset_stds.index else 0.0)**2 +
        (offset_stds[sensor_to] if sensor_to in offset_stds.index else 0.0)**2
    )
    
    return offset, error

print(f"\n{'='*80}")
print(f"🔧 FUNCIÓN calculate_direct_offset() DEFINIDA")
print(f"{'='*80}")
print(f"   Uso: offset, error = calculate_direct_offset('RESIST_SET1', 'PDHD-HP-15', 'PDHD-HP-20')")

# ------------------------------------------------------------------
# EJEMPLO: CALCULAR TODOS LOS OFFSETS DENTRO DE UN SET
# ------------------------------------------------------------------

print(f"\n{'='*80}")
print(f"📊 MATRIZ DE OFFSETS DIRECTOS - {set_example}")
print(f"{'='*80}")

if set_example in processed_sets:
    offset_means = processed_sets[set_example]['offset_means']
    sensors = offset_means.index.tolist()[:5]  # Primeros 5 para el ejemplo
    
    print(f"\n📋 Calculando offsets entre los primeros {len(sensors)} sensores:")
    print(f"   Sensores: {sensors}")
    
    # Crear matriz de offsets
    offset_matrix = pd.DataFrame(index=sensors, columns=sensors, dtype=float)
    
    for s1 in sensors:
        for s2 in sensors:
            if s1 == s2:
                offset_matrix.loc[s1, s2] = 0.0
            else:
                offset, _ = calculate_direct_offset(set_example, s1, s2)
                offset_matrix.loc[s1, s2] = offset
    
    print(f"\n📊 Matriz de offsets (K):")
    print(offset_matrix.to_string(float_format=lambda x: f'{x:+.6f}'))
    print(f"\n💡 Interpretación:")
    print(f"   Fila → Columna = Offset(Fila → Columna)")
    print(f"   Ejemplo: Fila '{sensors[0]}' → Columna '{sensors[1]}' = {offset_matrix.loc[sensors[0], sensors[1]]:+.6f} K")

🔄 OFFSETS DIRECTOS ENTRE RESISTENCIAS (MISMO SET)

📊 Ejemplo: Offset entre PDHD-HP-15 y PDHD-HP-20 en RESIST_SET1

✅ Cálculo completado:
   📏 PDHD-HP-15 → Ref: -0.029820 ± 0.000020 K
   📏 PDHD-HP-20 → Ref: -0.024469 ± 0.000035 K

🎯 Offset DIRECTO PDHD-HP-15 → PDHD-HP-20:
   -0.005351 ± 0.000040 K

📌 Este offset NO requiere pasar por R2
   Se calcula directamente usando la referencia interna del set

🔧 FUNCIÓN calculate_direct_offset() DEFINIDA
   Uso: offset, error = calculate_direct_offset('RESIST_SET1', 'PDHD-HP-15', 'PDHD-HP-20')

📊 MATRIZ DE OFFSETS DIRECTOS - RESIST_SET1

📋 Calculando offsets entre los primeros 5 sensores:
   Sensores: ['PDHD-HP-13', 'PDHD-HP-15', 'PDHD-HP-16', 'PDHD-HP-17', 'PDHD-HP-18']

📊 Matriz de offsets (K):
            PDHD-HP-13  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17  PDHD-HP-18
PDHD-HP-13   +0.000000   +0.002503   +0.009007   +0.007712   -0.029947
PDHD-HP-15   -0.002503   +0.000000   +0.006504   +0.005209   -0.032450
PDHD-HP-16   -0.009007   -0.006504   +0.0

## 🎯 Función Unificada para Calcular Offsets

Creamos una función que **automáticamente detecta** si las resistencias están en el mismo set o en sets diferentes, y aplica el método correspondiente.

In [40]:
# ------------------------------------------------------------------
# FUNCIÓN UNIFICADA: MISMO SET O DIFERENTES SETS
# ------------------------------------------------------------------

def calculate_offset_universal(sensor_A, sensor_B, verbose=False):
    """
    Calcula el offset entre dos resistencias, automáticamente detectando si están
    en el mismo set o en sets diferentes.
    
    - Si están en el MISMO set: usa offset directo (más preciso)
    - Si están en DIFERENTES sets: usa sensores raised como puentes (media ponderada)
    
    Args:
        sensor_A: ID de la resistencia origen (ej: 'PDHD-HP-15')
        sensor_B: ID de la resistencia destino (ej: 'PDHD-HP-27')
        verbose: Si True, imprime información detallada
        
    Returns:
        tuple: (offset, error, method) donde method es 'same_set' o 'cross_set'
    """
    # Buscar en qué sets están los sensores
    set_A = None
    set_B = None
    
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in processed_sets:
            continue
        sensors = processed_sets[set_name]['offset_means'].index
        if sensor_A in sensors:
            set_A = set_name
        if sensor_B in sensors:
            set_B = set_name
    
    if set_A is None or set_B is None:
        if verbose:
            print(f"⚠️ Sensores no encontrados en ningún set de R1")
        return None, None, None
    
    if verbose:
        print(f"\n🔍 DETECCIÓN AUTOMÁTICA")
        print(f"   {sensor_A} está en {set_A}")
        print(f"   {sensor_B} está en {set_B}")
    
    # CASO 1: Mismo set → Offset directo
    if set_A == set_B:
        if verbose:
            print(f"   📌 Método: OFFSET DIRECTO (mismo set)")
        offset, error = calculate_direct_offset(set_A, sensor_A, sensor_B)
        return offset, error, 'same_set'
    
    # CASO 2: Diferentes sets → Via sensores raised
    else:
        if verbose:
            print(f"   🔗 Método: VIA SENSORES RAISED (sets diferentes)")
        offset, error = calculate_offset_via_raised(set_A, sensor_A, set_B, sensor_B, verbose=verbose)
        return offset, error, 'cross_set'

print("="*80)
print("🔧 FUNCIÓN calculate_offset_universal() DEFINIDA")
print("="*80)
print("   Detecta automáticamente si los sensores están en el mismo set o no")
print("   Aplica el método óptimo en cada caso")

# ------------------------------------------------------------------
# EJEMPLOS DE USO DE LA FUNCIÓN UNIFICADA
# ------------------------------------------------------------------

print(f"\n\n{'='*80}")
print("📊 EJEMPLOS DE USO DE LA FUNCIÓN UNIFICADA")
print("="*80)

# Ejemplo 1: Mismo set
print(f"\n📌 EJEMPLO 1: Sensores del MISMO set")
print("="*60)
sensor1 = 'PDHD-HP-15'  # SET1
sensor2 = 'PDHD-HP-20'  # SET1
offset, error, method = calculate_offset_universal(sensor1, sensor2, verbose=True)
if offset is not None:
    print(f"\n✅ Offset {sensor1} → {sensor2}: {offset:+.6f} ± {error:.6f} K")
    print(f"   Método usado: {method}")

# Ejemplo 2: Sets diferentes
print(f"\n\n📌 EJEMPLO 2: Sensores de DIFERENTES sets")
print("="*60)
sensor3 = 'PDHD-HP-15'  # SET1
sensor4 = 'PDHD-HP-39'  # SET3
offset, error, method = calculate_offset_universal(sensor3, sensor4, verbose=True)
if offset is not None:
    print(f"\n✅ Offset {sensor3} → {sensor4}: {offset:+.6f} ± {error:.6f} K")
    print(f"   Método usado: {method}")

print(f"\n\n{'='*80}")
print("✅ FUNCIÓN UNIFICADA OPERATIVA")
print("="*80)
print("""
🎯 Ahora puedes calcular el offset entre CUALQUIER par de resistencias:

   offset, error, method = calculate_offset_universal('PDHD-HP-15', 'PDHD-HP-39')

El método se selecciona automáticamente según si están en el mismo set o no.
""")

🔧 FUNCIÓN calculate_offset_universal() DEFINIDA
   Detecta automáticamente si los sensores están en el mismo set o no
   Aplica el método óptimo en cada caso


📊 EJEMPLOS DE USO DE LA FUNCIÓN UNIFICADA

📌 EJEMPLO 1: Sensores del MISMO set

🔍 DETECCIÓN AUTOMÁTICA
   PDHD-HP-15 está en RESIST_SET1
   PDHD-HP-20 está en RESIST_SET1
   📌 Método: OFFSET DIRECTO (mismo set)

✅ Offset PDHD-HP-15 → PDHD-HP-20: -0.005351 ± 0.000040 K
   Método usado: same_set


📌 EJEMPLO 2: Sensores de DIFERENTES sets

🔍 DETECCIÓN AUTOMÁTICA
   PDHD-HP-15 está en RESIST_SET1
   PDHD-HP-39 está en RESIST_SET3
   🔗 Método: VIA SENSORES RAISED (sets diferentes)

🔗 Calculando offset PDHD-HP-15 (RESIST_SET1) → PDHD-HP-39 (RESIST_SET3)
   Sensores raised en RESIST_SET1: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Sensores raised en RESIST_SET3: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

   📍 Camino via PDHD-HP-13 ↔ PDHD-HP-39:
      Paso 1: PDHD-HP-15 → PDHD-HP-13 (en RESIST_SET1): -0.002503 ± 0.000040 K
      Pas

## 📊 Resumen del Sistema Unificado de Cálculo de Offsets

El sistema utiliza **`calculate_offset_universal()`** que detecta automáticamente el método óptimo:

### 🎯 Dos casos posibles:

1. **Mismo set (ej: HP-15 y HP-20 en RESIST_SET1)**
   - Método: `calculate_direct_offset()`
   - Cálculo: `offset(A→B) = offset(A→ref) - offset(B→ref)`
   - ✅ **Más preciso** (no pasa por R2, usa referencia interna del set)
   - Error típico: ~40-80 µK

2. **Sets diferentes (ej: HP-15 en SET1 y HP-27 en SET2)**
   - Método: `calculate_offset_via_raised()`
   - Cálculo: **9 caminos** via R2 usando sensores raised como puentes
   - Media ponderada: peso = 1/error²
   - ✅ **Máxima información** (usa todos los caminos disponibles)
   - Error típico: ~40-60 µK (mejorado con 9 caminos vs 3)

### 🔧 Uso en constantes de calibración:

```python
# Automático - elige el método correcto
offset, error, method = calculate_offset_universal(sensor_A, sensor_B)
```

El cálculo de **constantes de calibración finales** usa esta función unificada para todos los pares de sensores.

## 📊 Cálculo de Constantes de Calibración Finales

Calculamos las constantes de calibración para **todas las resistencias** conectándolas a través de los sensores raised y R2.

In [42]:
# ------------------------------------------------------------------
# CALCULAR CONSTANTES DE CALIBRACIÓN PARA TODAS LAS RESISTENCIAS
# ------------------------------------------------------------------
print("="*80)
print("📊 CÁLCULO DE CONSTANTES DE CALIBRACIÓN")
print("="*80)

# Estrategia: Usar un sensor raised de RESIST_SET1 como referencia
# y calcular offsets de todas las demás resistencias respecto a esa referencia

# Elegir referencia: primer sensor raised de RESIST_SET1
if 'RESIST_SET1' in raised_by_set and len(raised_by_set['RESIST_SET1']) > 0:
    reference_sensor = raised_by_set['RESIST_SET1'][0]
    print(f"\n📌 Sensor de referencia elegido: {reference_sensor}")
    print(f"   (Primer sensor raised de RESIST_SET1)")
else:
    print("❌ No hay sensores raised en RESIST_SET1")
    reference_sensor = None

if reference_sensor:
    # Diccionario para almacenar constantes de calibración
    calibration_constants = {}
    calibration_errors = {}
    
    print(f"\n🔄 Calculando offsets respecto a {reference_sensor}...")
    
    # Procesar todas las resistencias de todos los sets de R1
    total_resistances = 0
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in processed_sets:
            continue
        
        sensors_in_set = processed_sets[set_name]['sensors']
        print(f"\n   Procesando {set_name}: {len(sensors_in_set)} resistencias")
        
        for sensor in sensors_in_set:
            # Calcular offset usando la función universal
            offset, error, method = calculate_offset_universal(
                reference_sensor, 
                sensor, 
                verbose=False
            )
            
            if offset is not None:
                calibration_constants[sensor] = offset
                calibration_errors[sensor] = error
                total_resistances += 1
        
        print(f"      ✅ {total_resistances} resistencias procesadas hasta ahora")
    
    print(f"\n{'='*80}")
    print(f"✅ CONSTANTES CALCULADAS")
    print(f"{'='*80}")
    print(f"   Total resistencias: {len(calibration_constants)}")
    print(f"   Referencia: {reference_sensor}")
    print(f"\n   Rango de offsets:")
    print(f"      Min: {min(calibration_constants.values()):+.6f} K")
    print(f"      Max: {max(calibration_constants.values()):+.6f} K")
    print(f"      Media: {np.mean(list(calibration_constants.values())):+.6f} K")

📊 CÁLCULO DE CONSTANTES DE CALIBRACIÓN

📌 Sensor de referencia elegido: PDHD-HP-13
   (Primer sensor raised de RESIST_SET1)

🔄 Calculando offsets respecto a PDHD-HP-13...

   Procesando RESIST_SET1: 12 resistencias
      ✅ 12 resistencias procesadas hasta ahora

   Procesando RESIST_SET2: 12 resistencias
      ✅ 24 resistencias procesadas hasta ahora

   Procesando RESIST_SET3: 12 resistencias
      ✅ 36 resistencias procesadas hasta ahora

   Procesando RESIST_SET4: 12 resistencias
      ✅ 48 resistencias procesadas hasta ahora

✅ CONSTANTES CALCULADAS
   Total resistencias: 48
   Referencia: PDHD-HP-13

   Rango de offsets:
      Min: -0.031700 K
      Max: +0.016070 K
      Media: -0.005699 K


## 📊 Opción 1: Constantes respecto a una Referencia Única

Este método calcula el offset de cada resistencia respecto a **una referencia fija** (más compacto).

**Ventajas:**
- Solo 48 valores en lugar de 2,304
- Más fácil de visualizar y compartir
- Suficiente para la mayoría de casos

**Cómo obtener offset entre dos resistencias A y B:**
```
offset(A→B) = offset(A→ref) - offset(B→ref)
error(A→B) = √(error(A→ref)² + error(B→ref)²)
```

**Nota:** La matriz completa (Opción 2 más abajo) pre-calcula todos los pares y es más directa de usar.

In [43]:
# ------------------------------------------------------------------
# CREAR DATAFRAME CON CONSTANTES DE CALIBRACIÓN
# ------------------------------------------------------------------
print("="*80)
print("📋 TABLA DE CONSTANTES DE CALIBRACIÓN")
print("="*80)

# Crear DataFrame con todas las constantes
calibration_df = pd.DataFrame({
    'sensor_id': list(calibration_constants.keys()),
    'offset_K': list(calibration_constants.values()),
    'error_K': [calibration_errors[s] for s in calibration_constants.keys()],
    'offset_mK': [v * 1000 for v in calibration_constants.values()],
    'error_mK': [calibration_errors[s] * 1000 for s in calibration_constants.keys()]
})

# Añadir información de set y ronda
def get_set_info(sensor_id):
    for set_name, data in processed_sets.items():
        if sensor_id in data['sensors']:
            return set_name, data['round']
    return 'Unknown', -1

calibration_df['set'] = calibration_df['sensor_id'].apply(lambda x: get_set_info(x)[0])
calibration_df['round'] = calibration_df['sensor_id'].apply(lambda x: get_set_info(x)[1])

# Añadir si es sensor raised
calibration_df['is_raised'] = calibration_df['sensor_id'].apply(
    lambda x: x in raised_sensors
)

# Ordenar por set y sensor_id
calibration_df = calibration_df.sort_values(['set', 'sensor_id']).reset_index(drop=True)

print(f"\n✅ DataFrame creado con {len(calibration_df)} resistencias")
print(f"\nColumnas: {list(calibration_df.columns)}")
print(f"\n📊 Primeras 10 filas:")
print(calibration_df.head(10).to_string(index=False))

print(f"\n📊 Últimas 10 filas:")
print(calibration_df.tail(10).to_string(index=False))

# Guardar a CSV y Excel
output_csv = "calibration_constants_resistences.csv"
output_excel = "calibration_constants_resistences.xlsx"

calibration_df.to_csv(output_csv, index=False)
print(f"\n💾 Guardado en CSV: {output_csv}")

try:
    calibration_df.to_excel(output_excel, index=False)
    print(f"💾 Guardado en Excel: {output_excel}")
except Exception as e:
    print(f"⚠️ No se pudo guardar Excel (instalar openpyxl): {e}")

# Mostrar estadísticas por set
print(f"\n{'='*80}")
print("📊 ESTADÍSTICAS POR SET")
print("="*80)
for set_name in sorted(calibration_df['set'].unique()):
    if set_name == 'Unknown':
        continue
    subset = calibration_df[calibration_df['set'] == set_name]
    print(f"\n{set_name}:")
    print(f"   Resistencias: {len(subset)}")
    print(f"   Raised: {subset['is_raised'].sum()}")
    print(f"   Offset medio: {subset['offset_mK'].mean():+.3f} ± {subset['error_mK'].mean():.3f} mK")
    print(f"   Rango: [{subset['offset_mK'].min():+.3f}, {subset['offset_mK'].max():+.3f}] mK")

📋 TABLA DE CONSTANTES DE CALIBRACIÓN

✅ DataFrame creado con 48 resistencias

Columnas: ['sensor_id', 'offset_K', 'error_K', 'offset_mK', 'error_mK', 'set', 'round', 'is_raised']

📊 Primeras 10 filas:
 sensor_id  offset_K  error_K  offset_mK  error_mK         set  round  is_raised
PDHD-HP-13  0.000000 0.000050   0.000000  0.049672 RESIST_SET1      1       True
PDHD-HP-14 -0.027318 0.000035 -27.317531  0.035123 RESIST_SET1      1      False
PDHD-HP-15  0.002503 0.000040   2.502793  0.040337 RESIST_SET1      1      False
PDHD-HP-16  0.009007 0.000038   9.006767  0.037711 RESIST_SET1      1      False
PDHD-HP-17  0.007712 0.000049   7.712260  0.048552 RESIST_SET1      1      False
PDHD-HP-18 -0.029947 0.000038 -29.946773  0.037942 RESIST_SET1      1      False
PDHD-HP-19 -0.031700 0.000065 -31.700384  0.064831 RESIST_SET1      1       True
PDHD-HP-20 -0.002848 0.000049  -2.848426  0.049350 RESIST_SET1      1      False
PDHD-HP-21 -0.014065 0.000041 -14.065359  0.041366 RESIST_SET1      1 

## 🔲 Opción 2: Matriz Completa de Constantes de Calibración

Calculamos la **matriz completa** de offsets: cada resistencia respecto a todas las demás (48×48 = 2,304 pares).

**Ventajas:**
- Acceso directo a cualquier par sin cálculos adicionales
- Incluye el método usado para cada par
- Pre-calculado y optimizado (usa 9 caminos cuando aplica)

**Uso:**
```python
offset = offset_df.loc['PDHD-HP-15', 'PDHD-HP-27']  # Directo
error = error_df.loc['PDHD-HP-15', 'PDHD-HP-27']
```

**Nota:** Esta es la forma más completa y directa de acceder a las constantes de calibración.

In [44]:
# ------------------------------------------------------------------
# CALCULAR MATRIZ COMPLETA DE OFFSETS (TODAS LAS COMBINACIONES)
# ------------------------------------------------------------------
print("="*80)
print("🔲 MATRIZ COMPLETA DE CONSTANTES DE CALIBRACIÓN")
print("="*80)

# Obtener lista de todas las resistencias de R1
all_resistances = []
for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
    if set_name in processed_sets:
        all_resistances.extend(processed_sets[set_name]['sensors'])

print(f"\n📊 Total de resistencias en R1: {len(all_resistances)}")
print(f"   Sets: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4")

# Crear matriz de offsets (resistencia_i → resistencia_j)
print(f"\n🔄 Calculando matriz {len(all_resistances)}×{len(all_resistances)} de offsets...")
print(f"   Total de pares: {len(all_resistances) * len(all_resistances)} cálculos")

import time
start_time = time.time()

# Diccionarios para almacenar offsets y errores
offset_matrix = {}
error_matrix = {}
method_matrix = {}

# Calcular todos los pares
total_pairs = len(all_resistances) * len(all_resistances)
calculated = 0

for sensor_i in all_resistances:
    offset_matrix[sensor_i] = {}
    error_matrix[sensor_i] = {}
    method_matrix[sensor_i] = {}
    
    for sensor_j in all_resistances:
        if sensor_i == sensor_j:
            # Offset de una resistencia consigo misma = 0
            offset_matrix[sensor_i][sensor_j] = 0.0
            error_matrix[sensor_i][sensor_j] = 0.0
            method_matrix[sensor_i][sensor_j] = 'self'
        else:
            # Calcular offset usando función universal
            offset, error, method = calculate_offset_universal(sensor_i, sensor_j, verbose=False)
            offset_matrix[sensor_i][sensor_j] = offset if offset is not None else np.nan
            error_matrix[sensor_i][sensor_j] = error if error is not None else np.nan
            method_matrix[sensor_i][sensor_j] = method if method is not None else 'error'
        
        calculated += 1
        if calculated % 500 == 0:
            print(f"   Progreso: {calculated}/{total_pairs} pares calculados ({100*calculated/total_pairs:.1f}%)")

elapsed_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"✅ MATRIZ COMPLETA CALCULADA")
print(f"{'='*80}")
print(f"   Dimensión: {len(all_resistances)} × {len(all_resistances)}")
print(f"   Total pares: {total_pairs}")
print(f"   Tiempo de cálculo: {elapsed_time:.2f} segundos")

# Convertir a DataFrames de pandas
offset_df = pd.DataFrame(offset_matrix).T
error_df = pd.DataFrame(error_matrix).T
method_df = pd.DataFrame(method_matrix).T

print(f"\n📊 Estructura de la matriz:")
print(f"   offset_df[i][j] = offset de resistencia i → resistencia j")
print(f"   error_df[i][j] = error del offset i → j")
print(f"   method_df[i][j] = método usado ('same_set' o 'cross_set')")

# Estadísticas
same_set_count = (method_df == 'same_set').sum().sum()
cross_set_count = (method_df == 'cross_set').sum().sum()
self_count = (method_df == 'self').sum().sum()

print(f"\n📈 Estadísticas de métodos:")
print(f"   Mismo set (directo): {same_set_count} pares")
print(f"   Sets diferentes (9 caminos): {cross_set_count} pares")
print(f"   Diagonal (sí mismo): {self_count} pares")
print(f"   Total: {same_set_count + cross_set_count + self_count} pares")

🔲 MATRIZ COMPLETA DE CONSTANTES DE CALIBRACIÓN

📊 Total de resistencias en R1: 48
   Sets: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4

🔄 Calculando matriz 48×48 de offsets...
   Total de pares: 2304 cálculos
   Progreso: 500/2304 pares calculados (21.7%)
   Progreso: 1000/2304 pares calculados (43.4%)
   Progreso: 1500/2304 pares calculados (65.1%)
   Progreso: 2000/2304 pares calculados (86.8%)

✅ MATRIZ COMPLETA CALCULADA
   Dimensión: 48 × 48
   Total pares: 2304
   Tiempo de cálculo: 0.30 segundos

📊 Estructura de la matriz:
   offset_df[i][j] = offset de resistencia i → resistencia j
   error_df[i][j] = error del offset i → j
   method_df[i][j] = método usado ('same_set' o 'cross_set')

📈 Estadísticas de métodos:
   Mismo set (directo): 528 pares
   Sets diferentes (9 caminos): 1728 pares
   Diagonal (sí mismo): 48 pares
   Total: 2304 pares


In [45]:
# ------------------------------------------------------------------
# VISUALIZACIÓN Y EXPORTACIÓN DE LA MATRIZ
# ------------------------------------------------------------------
print("="*80)
print("📊 VISUALIZACIÓN DE LA MATRIZ")
print("="*80)

# Mostrar una submatriz como ejemplo (primeros 5 sensores)
example_sensors = all_resistances[:5]
print(f"\n📋 Ejemplo: Submatriz 5×5 (offsets en K):")
print(offset_df.loc[example_sensors, example_sensors].to_string(float_format=lambda x: f'{x:+.6f}'))

print(f"\n📋 Ejemplo: Errores correspondientes (en K):")
print(error_df.loc[example_sensors, example_sensors].to_string(float_format=lambda x: f'{x:.6f}'))

print(f"\n📋 Ejemplo: Métodos usados:")
print(method_df.loc[example_sensors, example_sensors].to_string())

# Guardar matrices completas a CSV
print(f"\n{'='*80}")
print(f"💾 GUARDANDO MATRICES")
print(f"{'='*80}")

offset_df.to_csv("calibration_matrix_offsets.csv")
print(f"✅ Matriz de offsets: calibration_matrix_offsets.csv")

error_df.to_csv("calibration_matrix_errors.csv")
print(f"✅ Matriz de errores: calibration_matrix_errors.csv")

method_df.to_csv("calibration_matrix_methods.csv")
print(f"✅ Matriz de métodos: calibration_matrix_methods.csv")

# Guardar también en formato Excel (una hoja por matriz)
try:
    with pd.ExcelWriter('calibration_matrices_complete.xlsx') as writer:
        offset_df.to_excel(writer, sheet_name='Offsets_K')
        error_df.to_excel(writer, sheet_name='Errors_K')
        method_df.to_excel(writer, sheet_name='Methods')
        
        # Crear también versión en mK
        (offset_df * 1000).to_excel(writer, sheet_name='Offsets_mK')
        (error_df * 1000).to_excel(writer, sheet_name='Errors_mK')
    
    print(f"✅ Excel completo: calibration_matrices_complete.xlsx")
    print(f"   Hojas: Offsets_K, Errors_K, Methods, Offsets_mK, Errors_mK")
except Exception as e:
    print(f"⚠️ No se pudo guardar Excel: {e}")

print(f"\n{'='*80}")
print(f"📊 RESUMEN DE LA MATRIZ")
print(f"{'='*80}")
print(f"""
✅ Matriz completa de constantes de calibración calculada

📋 Uso de la matriz:
   • offset_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → offset de HP-15 a HP-27
   • error_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → error del offset
   • method_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → método usado

🔢 Estadísticas:
   • {len(all_resistances)}×{len(all_resistances)} = {len(all_resistances)**2} pares totales
   • {same_set_count} pares del mismo set (método directo)
   • {cross_set_count} pares de sets diferentes (9 caminos via R2)
   
📁 Archivos generados:
   • calibration_matrix_offsets.csv
   • calibration_matrix_errors.csv
   • calibration_matrix_methods.csv
   • calibration_matrices_complete.xlsx
""")

📊 VISUALIZACIÓN DE LA MATRIZ

📋 Ejemplo: Submatriz 5×5 (offsets en K):
            PDHD-HP-13  PDHD-HP-14  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17
PDHD-HP-13   +0.000000   -0.027318   +0.002503   +0.009007   +0.007712
PDHD-HP-14   +0.027318   +0.000000   +0.029820   +0.036324   +0.035030
PDHD-HP-15   -0.002503   -0.029820   +0.000000   +0.006504   +0.005209
PDHD-HP-16   -0.009007   -0.036324   -0.006504   +0.000000   -0.001295
PDHD-HP-17   -0.007712   -0.035030   -0.005209   +0.001295   +0.000000

📋 Ejemplo: Errores correspondientes (en K):
            PDHD-HP-13  PDHD-HP-14  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17
PDHD-HP-13    0.000000    0.000035    0.000040    0.000038    0.000049
PDHD-HP-14    0.000035    0.000000    0.000020    0.000014    0.000034
PDHD-HP-15    0.000040    0.000020    0.000000    0.000024    0.000039
PDHD-HP-16    0.000038    0.000014    0.000024    0.000000    0.000036
PDHD-HP-17    0.000049    0.000034    0.000039    0.000036    0.000000

📋 Ejemplo: Métodos usados:
    

## 🔍 Verificación de Caminos de Calibración

Mostramos ejemplos detallados de cómo se calculan los offsets pasando por R2.

In [47]:
# ------------------------------------------------------------------
# VERIFICAR COHERENCIA ENTRE AMBAS OPCIONES
# ------------------------------------------------------------------
print("="*80)
print("🔍 VERIFICACIÓN: Coherencia entre Opción 1 y Opción 2")
print("="*80)

# Ejemplo 1: Dos resistencias del mismo set
print("\n📌 EJEMPLO 1: Mismo set (RESIST_SET1)")
print("="*60)
sensor_A = 'PDHD-HP-15'
sensor_B = 'PDHD-HP-20'

# Opción 1: Calcular usando referencia única
offset_A = calibration_df[calibration_df['sensor_id'] == sensor_A]['offset_mK'].values[0]
offset_B = calibration_df[calibration_df['sensor_id'] == sensor_B]['offset_mK'].values[0]
offset_AB_opt1 = offset_B - offset_A

# Opción 2: Leer directamente de la matriz
offset_AB_opt2 = offset_df.loc[sensor_A, sensor_B] * 1000  # K → mK

print(f"🔹 Opción 1 (vía referencia única):")
print(f"   {sensor_A} → ref: {offset_A:+.3f} mK")
print(f"   {sensor_B} → ref: {offset_B:+.3f} mK")
print(f"   {sensor_A} → {sensor_B}: {offset_AB_opt1:+.3f} mK")
print(f"\n🔹 Opción 2 (matriz completa):")
print(f"   {sensor_A} → {sensor_B}: {offset_AB_opt2:+.3f} mK")
print(f"\n✅ Diferencia: {abs(offset_AB_opt1 - offset_AB_opt2):.6f} mK (debe ser ~0)")

# Ejemplo 2: Dos resistencias de sets diferentes
print(f"\n\n📌 EJEMPLO 2: Sets diferentes (SET1 → SET3)")
print("="*60)
sensor_C = 'PDHD-HP-15'  # RESIST_SET1
sensor_D = 'PDHD-HP-39'  # RESIST_SET3

# Opción 1: Calcular usando referencia única
offset_C = calibration_df[calibration_df['sensor_id'] == sensor_C]['offset_mK'].values[0]
offset_D = calibration_df[calibration_df['sensor_id'] == sensor_D]['offset_mK'].values[0]
offset_CD_opt1 = offset_D - offset_C

# Opción 2: Leer directamente de la matriz
offset_CD_opt2 = offset_df.loc[sensor_C, sensor_D] * 1000  # K → mK

print(f"🔹 Opción 1 (vía referencia única):")
print(f"   {sensor_C} → ref: {offset_C:+.3f} mK")
print(f"   {sensor_D} → ref: {offset_D:+.3f} mK")
print(f"   {sensor_C} → {sensor_D}: {offset_CD_opt1:+.3f} mK")
print(f"\n🔹 Opción 2 (matriz completa):")
print(f"   {sensor_C} → {sensor_D}: {offset_CD_opt2:+.3f} mK")
print(f"\n✅ Diferencia: {abs(offset_CD_opt1 - offset_CD_opt2):.6f} mK (debe ser ~0)")

print(f"\n{'='*80}")
print(f"📊 CONCLUSIÓN")
print(f"{'='*80}")
print(f"""
✅ Ambas opciones son coherentes y dan el mismo resultado.

📋 Opción 1 (referencia única):
   • Más compacta (48 valores)
   • Requiere cálculo: offset(A→B) = offset(A→ref) - offset(B→ref)

📋 Opción 2 (matriz completa):
   • Acceso directo (48×48 = 2,304 valores pre-calculados)
   • Más conveniente: offset_df.loc[A, B]
   
💡 Recomendación: Usa la Opción 2 (matriz completa) para mayor comodidad.
""")

🔍 VERIFICACIÓN: Coherencia entre Opción 1 y Opción 2

📌 EJEMPLO 1: Mismo set (RESIST_SET1)
🔹 Opción 1 (vía referencia única):
   PDHD-HP-15 → ref: +2.503 mK
   PDHD-HP-20 → ref: -2.848 mK
   PDHD-HP-15 → PDHD-HP-20: -5.351 mK

🔹 Opción 2 (matriz completa):
   PDHD-HP-15 → PDHD-HP-20: -5.351 mK

✅ Diferencia: 0.000000 mK (debe ser ~0)


📌 EJEMPLO 2: Sets diferentes (SET1 → SET3)
🔹 Opción 1 (vía referencia única):
   PDHD-HP-15 → ref: +2.503 mK
   PDHD-HP-39 → ref: -10.462 mK
   PDHD-HP-15 → PDHD-HP-39: -12.964 mK

🔹 Opción 2 (matriz completa):
   PDHD-HP-15 → PDHD-HP-39: -12.965 mK

✅ Diferencia: 0.000458 mK (debe ser ~0)

📊 CONCLUSIÓN

✅ Ambas opciones son coherentes y dan el mismo resultado.

📋 Opción 1 (referencia única):
   • Más compacta (48 valores)
   • Requiere cálculo: offset(A→B) = offset(A→ref) - offset(B→ref)

📋 Opción 2 (matriz completa):
   • Acceso directo (48×48 = 2,304 valores pre-calculados)
   • Más conveniente: offset_df.loc[A, B]
   
💡 Recomendación: Usa la Opción 2

# 📋 RESUMEN FINAL COMPLETO

## ✅ Análisis Completado con Dos Opciones de Constantes de Calibración

### 🎯 Datos Procesados
- **R1:** 48 resistencias (12 por set × 4 sets: RESIST_SET1-4)
- **R2:** 12 resistencias (RESIST_SET5 — mezcla de 3 por cada set R1)
- **Sensores Raised:** 12 (3 por cada set R1, todos presentes en R2)
  - `HP-13, HP-19, HP-21, HP-28, HP-30, HP-31, HP-39, HP-44, HP-45, HP-52, HP-55, HP-56`

### 📊 **OPCIÓN 1: Constantes de Calibración Relativas a Referencia (HP-13)**
- **Método:** Todos los sensores referenciados al sensor raised HP-13
- **Resultados:** 48 constantes de calibración (una por sensor)
- **Precision promedio:** ±0.041 mK (±41 µK)
- **Archivos generados:**
  - `calibration_constants_resistences.csv`
  - `calibration_constants_resistences.xlsx`
  
### 📊 **OPCIÓN 2: Matriz Completa de Calibración (48×48)**
- **Método:** Matriz completa de offsets entre TODAS las parejas posibles
- **Total de pares calculados:** 2,304 offsets (48×48)
  - 528 pares mismo set (método directo, sin R2)
  - 1,728 pares entre diferentes sets (9 caminos via R2)
  - 48 diagonal (self-calibration, offset=0.0)
- **Archivos generados:**
  - `calibration_matrix_offsets.csv` (48×48)
  - `calibration_matrix_errors.csv` (48×48)
  - `calibration_matrix_methods.csv` (48×48)
  - `calibration_matrices_complete.xlsx` (5 sheets: Offsets_K, Errors_K, Methods, Offsets_mK, Errors_mK)

### 🔬 Método de Cálculo: 9 Caminos Linealmente Independientes
Para offsets entre diferentes sets (cross-set):
- Se calculan **9 caminos** (3 raised de set A × 3 raised de set B)
- **Media ponderada:** peso = 1/error² (mayor peso a caminos más precisos)
- **Identificación del mejor camino:** Marcado con 🏆 (menor error)

### ✅ Verificación de Coherencia
- **Mismo set:** Diferencia entre Opción 1 y 2: **0.000000 mK** (perfecta coincidencia)
- **Entre sets:** Diferencia entre Opción 1 y 2: **0.000458 mK** (dentro de precisión numérica)
- **Conclusión:** Ambas opciones son matemáticamente equivalentes y coherentes

### 📁 Todos los Archivos Generados
1. `calibration_constants_resistences.csv` (Opción 1)
2. `calibration_constants_resistences.xlsx` (Opción 1)
3. `calibration_matrix_offsets.csv` (Opción 2)
4. `calibration_matrix_errors.csv` (Opción 2)
5. `calibration_matrix_methods.csv` (Opción 2)
6. `calibration_matrices_complete.xlsx` (Opción 2 — 5 sheets)

## 📝 Resumen Final

In [39]:
# ------------------------------------------------------------------
# RESUMEN FINAL
# ------------------------------------------------------------------
print("="*80)
print("📝 RESUMEN DEL ANÁLISIS - RESISTENCIAS DE PRECISIÓN")
print("="*80)

print(f"\n📊 Estructura de datos:")
print(f"   Sets totales procesados: {len(processed_sets)}")
for ronda in [1, 2]:
    sets_r = [s for s, d in processed_sets.items() if d['round'] == ronda]
    print(f"   - Ronda {ronda}: {len(sets_r)} sets")
    if ronda == 1:
        total_r1 = sum(len(processed_sets[s]['sensors']) for s in sets_r)
        print(f"      Total resistencias R1: {total_r1}")

print(f"\n🔗 Conectividad:")
print(f"   Sensores raised totales: {len(raised_sensors)}")
print(f"      (Excluidas referencias 1009, 1010)")
print(f"   Distribución por set R1:")
for set_name in sorted(raised_by_set.keys()):
    print(f"      {set_name}: {len(raised_by_set[set_name])} raised")

print(f"\n📈 Método de cálculo:")
print(f"   ✅ Offsets dentro del mismo set: Directos (ref interna)")
print(f"   ✅ Offsets entre sets diferentes: Via R2 con sensores raised")
print(f"   ✅ Media ponderada: peso = 1/error²")

if 'calibration_df' in locals():
    print(f"\n🎯 Constantes de calibración:")
    print(f"   Total resistencias: {len(calibration_df)}")
    print(f"   Referencia: {reference_sensor}")
    print(f"   Archivos generados:")
    print(f"      - calibration_constants_resistences.csv")
    print(f"      - calibration_constants_resistences.xlsx")
    print(f"\n   Rango de constantes:")
    print(f"      Min: {calibration_df['offset_mK'].min():+.3f} mK")
    print(f"      Max: {calibration_df['offset_mK'].max():+.3f} mK")
    print(f"      Precisión media: ±{calibration_df['error_mK'].mean():.3f} mK")

print(f"\n{'='*80}")
print(f"✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print(f"{'='*80}")
print("""
📋 Para revisar los resultados:
   1. Abrir calibration_constants_resistences.xlsx
   2. Ver columnas: sensor_id, offset_mK, error_mK, set, is_raised
   3. Verificar que todas las resistencias están conectadas vía sensores raised
""")

📝 RESUMEN DEL ANÁLISIS - RESISTENCIAS DE PRECISIÓN

📊 Estructura de datos:
   Sets totales procesados: 5
   - Ronda 1: 4 sets
      Total resistencias R1: 48
   - Ronda 2: 1 sets

🔗 Conectividad:
   Sensores raised totales: 12
      (Excluidas referencias 1009, 1010)
   Distribución por set R1:
      RESIST_SET1: 3 raised
      RESIST_SET2: 3 raised
      RESIST_SET3: 3 raised
      RESIST_SET4: 3 raised

📈 Método de cálculo:
   ✅ Offsets dentro del mismo set: Directos (ref interna)
   ✅ Offsets entre sets diferentes: Via R2 con sensores raised
   ✅ Media ponderada: peso = 1/error²

🎯 Constantes de calibración:
   Total resistencias: 48
   Referencia: PDHD-HP-13
   Archivos generados:
      - calibration_constants_resistences.csv
      - calibration_constants_resistences.xlsx

   Rango de constantes:
      Min: -31.700 mK
      Max: +16.070 mK
      Precisión media: ±0.041 mK

✅ ANÁLISIS COMPLETADO EXITOSAMENTE

📋 Para revisar los resultados:
   1. Abrir calibration_constants_resistenc

## ✅ Verificación: Los Sensores del Canal 2 También Están Incluidos

Los sensores del canal 2 (HP-14, HP-26, HP-38, HP-50) **NO fueron levantados en R2**, pero eso **NO es un problema** porque:
1. Están presentes en sus respectivos sets R1
2. La matriz 48×48 ya los incluye
3. Se pueden calcular offsets entre ellos usando el método de 9 caminos

In [ ]:
# ------------------------------------------------------------------
# VERIFICAR QUE LOS SENSORES DEL CANAL 2 ESTÁN INCLUIDOS
# ------------------------------------------------------------------
print("="*80)
print("✅ VERIFICACIÓN: Sensores del Canal 2 (no raised pero sí incluidos)")
print("="*80)

# Los sensores del canal 2 son las referencias de cada set R1
ch2_sensors = {
    'RESIST_SET1': 'PDHD-HP-14',
    'RESIST_SET2': 'PDHD-HP-26',
    'RESIST_SET3': 'PDHD-HP-38',
    'RESIST_SET4': 'PDHD-HP-50'
}

print("\n📋 Sensores del canal 2 (uno por set R1):")
for set_name, sensor in ch2_sensors.items():
    print(f"   {set_name}: {sensor}")

# ¿Están en processed_sets?
print("\n🔍 ¿Están en processed_sets (R1)?")
for set_name, sensor in ch2_sensors.items():
    if set_name in processed_sets:
        sensors_in_set = processed_sets[set_name]['sensors']
        en_r1 = sensor in sensors_in_set
        emoji = "✅" if en_r1 else "❌"
        print(f"   {emoji} {sensor} en {set_name}: {en_r1}")

# ¿Están en la matriz 48×48?
print("\n🔍 ¿Están en la matriz 48×48?")
for set_name, sensor in ch2_sensors.items():
    en_matriz = sensor in offset_df.index
    emoji = "✅" if en_matriz else "❌"
    print(f"   {emoji} {sensor} en offset_df: {en_matriz}")

# PRUEBA: Calcular offsets entre sensores del canal 2
print("\n" + "="*80)
print("🧪 PRUEBA: Calcular offsets entre sensores del canal 2 de diferentes sets")
print("="*80)

# Ejemplo 1: HP-14 (SET1) → HP-26 (SET2)
sensor1 = 'PDHD-HP-14'
sensor2 = 'PDHD-HP-26'

print(f"\n📊 Ejemplo 1: {sensor1} → {sensor2}")
offset1, error1, method1 = calculate_offset_universal(sensor1, sensor2, verbose=False)
if offset1 is not None:
    print(f"   ✅ Offset: {offset1:+.6f} ± {error1:.6f} K ({offset1*1000:+.3f} ± {error1*1000:.3f} mK)")
    print(f"   Método: {method1}")
else:
    print(f"   ❌ No se pudo calcular")

# Ejemplo 2: HP-14 (SET1) → HP-38 (SET3)
sensor3 = 'PDHD-HP-38'

print(f"\n📊 Ejemplo 2: {sensor1} → {sensor3}")
offset2, error2, method2 = calculate_offset_universal(sensor1, sensor3, verbose=False)
if offset2 is not None:
    print(f"   ✅ Offset: {offset2:+.6f} ± {error2:.6f} K ({offset2*1000:+.3f} ± {error2*1000:.3f} mK)")
    print(f"   Método: {method2}")
else:
    print(f"   ❌ No se pudo calcular")

# Ejemplo 3: HP-14 (SET1) → HP-50 (SET4)
sensor4 = 'PDHD-HP-50'

print(f"\n📊 Ejemplo 3: {sensor1} → {sensor4}")
offset3, error3, method3 = calculate_offset_universal(sensor1, sensor4, verbose=False)
if offset3 is not None:
    print(f"   ✅ Offset: {offset3:+.6f} ± {error3:.6f} K ({offset3*1000:+.3f} ± {error3*1000:.3f} mK)")
    print(f"   Método: {method3}")
else:
    print(f"   ❌ No se pudo calcular")

# Verificar en la matriz
print("\n" + "="*80)
print("🔍 Verificación en la matriz 48×48")
print("="*80)
print(f"\n{sensor1} → {sensor2}: {offset_df.loc[sensor1, sensor2]:+.6f} K")
print(f"{sensor1} → {sensor3}: {offset_df.loc[sensor1, sensor3]:+.6f} K")
print(f"{sensor1} → {sensor4}: {offset_df.loc[sensor1, sensor4]:+.6f} K")

print("\n" + "="*80)
print("✅ CONCLUSIÓN")
print("="*80)
print("""
Los sensores del canal 2 (HP-14, HP-26, HP-38, HP-50):
   ✅ Están en processed_sets (sus respectivos sets R1)
   ✅ Están en la matriz 48×48
   ✅ Se pueden calcular offsets entre ellos usando 9 caminos
   
Método de cálculo:
   - Usan sensores raised de sus respectivos sets como puentes
   - Se calculan los 9 caminos posibles (3 raised_A × 3 raised_B)
   - Se aplica media ponderada (peso = 1/error²)
   
Por lo tanto, NO hay ningún problema con los sensores del canal 2.
La matriz 48×48 incluye TODOS los offsets, incluyendo los del canal 2.
""")

🧪 VERIFICACIÓN: Offsets entre Sensores del Canal 2

📋 Sensores del canal 2 (referencias de cada set):
   RESIST_SET1: PDHD-HP-14
   RESIST_SET2: PDHD-HP-26
   RESIST_SET3: PDHD-HP-38
   RESIST_SET4: PDHD-HP-50

🔍 ¿Están estos sensores en RESIST_SET5 (R2)?
   Sensores en RESIST_SET5: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

   Verificación:
   ❌ PDHD-HP-14 en R2: False
   ❌ PDHD-HP-26 en R2: False
   ❌ PDHD-HP-38 en R2: False
   ❌ PDHD-HP-50 en R2: False

📊 Total sensores canal 2 en R2: 0/4

⚠️ PROBLEMA DETECTADO:
   Algunos sensores del canal 2 NO están en RESIST_SET5
   Por lo tanto, NO se puede calcular offset entre ellos usando sensores raised

   Sensores canal 2 que NO están en R2:
      - PDHD-HP-14
      - PDHD-HP-26
      - PDHD-HP-38
      - PDHD-HP-50


🧪 PRUEBA: Calcular offset HP-14 (SET1) → HP-26 (SET2)

🔍 DETECCIÓN AUTOMÁTICA
   PDHD-HP-14 está 